# Recovery modeling and tail risk

Compare **constant**, **market-correlated**, and **market-standard stochastic** recovery via `RecoverySpec` / `RecoveryModel`, then stress **portfolio losses** when **LGD co-moves** with the same systematic factor as **defaults**.

## Concept

**Recovery** (or **LGD = 1 − R**) need not be fixed. **Market-correlated** recovery makes **expected recovery** depend on the **systematic** state: in bad markets, recoveries often **fall**, so **defaults** and **LGD** can **worsen together** — a **positive feedback** in the tail.

`RecoveryModel.conditional_recovery(Z)` and **`conditional_lgd(Z)`** describe that **systematic** slice. **Stochastic** recovery specs add **volatility** around the mean structure; the **market-standard** preset embeds a **negative** correlation to the factor (recovery tends to drop when the factor is **high** in the model’s convention — always read **`model_name`** and validate signs on your curve).

We first **tabulate** **conditional** recovery/LGD across $Z \in [-3, 3]$, then re-use the **portfolio simulation** engine with **factor-dependent LGD** on default to show **VaR/ES** **inflation** versus **constant** LGD.

## API walkthrough

- **`RecoverySpec.constant(rate)`** → **`.build()`** — fixed recovery; **LGD** constant in $Z$.
- **`RecoverySpec.market_correlated(mean, vol, correlation)`** — **Andersen–Sidenius**-style systematic recovery; **`conditional_lgd(Z)`** varies with $Z$.
- **`RecoverySpec.market_standard_stochastic()`** — preset **mean/vol/correlation** calibration.

All specs expose **`expected_recovery`**, **`recovery_volatility`**, and **`is_stochastic`** after **`.build()`**.

In [ ]:
from finstack.correlation import RecoverySpec

rec_const = RecoverySpec.constant(0.40).build()
rec_corr = RecoverySpec.market_correlated(0.40, 0.10, 0.50).build()
rec_stoch = RecoverySpec.market_standard_stochastic().build()

print(f"Constant: {rec_const}")
print(f"Market-correlated: {rec_corr}, vol={rec_corr.recovery_volatility:.4f}")
print(f"Market standard stochastic: {rec_stoch}, vol={rec_stoch.recovery_volatility:.4f}")
print(
    f"Unconditional expected recovery: const={rec_const.expected_recovery:.4f}, "
    f"corr={rec_corr.expected_recovery:.4f}, mkt_std={rec_stoch.expected_recovery:.4f}"
)

**Conditional** **recovery** and **LGD** across **market** $Z \in \{-3, -2, \ldots, 3\}$.

In [ ]:
print("Z    | R_const | LGD_c | R_corr | LGD_corr | R_mktstd | LGD_mktstd")
for z in range(-3, 4):
    zf = float(z)
    print(
        f"{zf:4.1f} | {rec_const.conditional_recovery(zf):7.4f} | "
        f"{rec_const.conditional_lgd(zf):5.4f} | {rec_corr.conditional_recovery(zf):6.4f} | "
        f"{rec_corr.conditional_lgd(zf):8.4f} | {rec_stoch.conditional_recovery(zf):8.4f} | "
        f"{rec_stoch.conditional_lgd(zf):10.4f}"
    )
print("Bad states (negative Z in many credit conventions) often lift LGD when recovery is market-linked.")

## Examples

Reuse the **10-name** portfolio from the default simulation notebook: same **PDs**, **notionals**, **Gaussian** copula, **asset correlation 0.20**, **10,000** paths, **`random.seed(42)`**. On each default, apply **`conditional_lgd(Z)`** for the **active** recovery model (constant case matches **0.60** LGD).

Print **mean**, **VaR(99%)**, and **ES(99%)** for each recovery assumption.

In [ ]:
import math
import random

from finstack.correlation import CopulaSpec
from finstack.core.math.special_functions import standard_normal_inv_cdf


def var_and_es(losses: list[float], alpha: float = 0.99) -> tuple[float, float]:
    xs = sorted(losses)
    n = len(xs)
    idx = min(n - 1, max(0, math.ceil(alpha * n) - 1))
    var_v = xs[idx]
    tail = [x for x in xs if x >= var_v]
    es_v = sum(tail) / len(tail) if tail else var_v
    return var_v, es_v


def simulate_portfolio(rec_model, n_sims: int, seed: int) -> list[float]:
    rng = random.Random(seed)
    gauss = CopulaSpec.gaussian().build()
    losses: list[float] = []
    for _ in range(n_sims):
        z = rng.gauss(0.0, 1.0)
        pl = 0.0
        for i in range(n_names):
            c_i = standard_normal_inv_cdf(pds[i])
            cond_pd = gauss.conditional_default_prob(c_i, [z], rho)
            if rng.random() < cond_pd:
                lgd_i = rec_model.conditional_lgd(z)
                pl += notionals[i] * lgd_i
        losses.append(pl)
    return losses


n_names = 10
pds = [0.02, 0.03, 0.05, 0.02, 0.04, 0.03, 0.06, 0.01, 0.03, 0.05]
notionals = [10_000_000.0] * n_names
rho = 0.20
n_sims = 10_000

models = [
    ("constant_40pct_R", rec_const),
    ("market_correlated", rec_corr),
    ("market_standard_stoch", rec_stoch),
]

print(f"Portfolio sim: {n_sims} paths, seed=42, rho={rho}")
for label, m in models:
    ls = simulate_portfolio(m, n_sims, 42)
    mean_v = sum(ls) / n_sims
    var_v, es_v = var_and_es(ls)
    print(
        f"{label:22} mean={mean_v:12,.0f}  VaR99={var_v:12,.0f}  ES99={es_v:12,.0f}"
    )

base = simulate_portfolio(rec_const, n_sims, 42)
mc = simulate_portfolio(rec_corr, n_sims, 42)
var_b, es_b = var_and_es(base)
var_m, es_m = var_and_es(mc)
print(
    f"Tail amplification (market-corr vs constant): VaR99 {((var_m / var_b) - 1) * 100:+.2f}%, "
    f"ES99 {((es_m / es_b) - 1) * 100:+.2f}%"
)

## Takeaways

- **Three** recovery styles — **constant**, **market-correlated**, **market-standard stochastic** — share the same **`RecoveryModel`** surface: inspect **`conditional_recovery` / `conditional_lgd`** along $Z$.
- When **LGD rises in bad systematic states** alongside **higher conditional PDs**, **portfolio** **tail** metrics (**VaR**, **ES**) can **exceed** the **constant-LGD** case even if **unconditional** mean recovery is **similar**.
- **Recovery–default** interaction is a **modeling choice**; always align **sign conventions** for $Z$ with your **copula** and **recovery** implementation.
- For **full** structured-credit workflows, combine with **`clo_tranche_modeling.ipynb`** (capital structure) and **`portfolio_default_simulation.ipynb`** (copula simulation).